# MLIP example: Well-tempered metadynamics — NaCl dissociation in water

**Before you begin: Note that this notebook runs a simulation that takes around 20–30 minutes on both CPU and GPU (tested on a 40GB NVIDIA H100), using the respective config values provided in Section 4.**

In this notebook we demonstrate **well-tempered metadynamics** using the dissociation
of a sodium chloride (NaCl) ion pair in explicit water as the example system.

NaCl dissociation in water is the metadynamics benchmark introduced in the
original metadynamics paper by Laio & Parrinello (2002) [1]. The system exhibits two
metastable states: the **contact ion pair** (CIP, ~2.7 Å) and the **solvent-separated
ion pair** (SSIP, ~5.0 Å). These two states are separated by a desolvation barrier of a few kcal/mol that
is too high for plain molecular dynamics (MD) to sample efficiently at 300 K, but not so high that a metadynamics run cannot converge easily.

Following [1], we use **two collective variables (CVs)**:
1. **Na–Cl distance** — the primary reaction coordinate for ion-pair dissociation
2. **Coordination number of Na with water oxygens** — captures the rearrangement of
   the first solvation shell as Cl⁻ leaves and water molecules move in to coordinate Na⁺

Using both CVs is essential: the distance alone cannot distinguish configurations where
the solvation shell has or has not rearranged around Na⁺, which are kinetically separated
states on the free-energy surface.

In our library we have implemented **well-tempered metadynamics** [2] in which hill heights are rescaled to ensure
convergence of the free-energy estimate.

---

[1] Laio, A. & Parrinello, M. *Escaping free-energy minima.* PNAS **99**, 12562–12566 (2002). https://doi.org/10.1073/pnas.202427399

[2] Barducci, A., Bussi, G. & Parrinello, M. *Well-tempered metadynamics: a smoothly converging and tunable free-energy method.* PRL **100**, 020603 (2008). https://doi.org/10.1103/PhysRevLett.100.020603

**Install and logging setup**

In [ ]:
%pip install "mlip[cuda]" huggingface_hub

# Use this instead for installation without GPU:
# %pip install mlip huggingface_hub

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, force=True, format="%(levelname)s - %(message)s"
)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="advanced_simulation/*",
    local_dir="",
)

In [ ]:
import jax

print(jax.devices())

## 1. Download and load a pre-trained MLIP model

In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="visnet_organics_02.zip",
    local_dir="pretrained_models/",
)

# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="mace_organics_02.zip",
#     local_dir="pretrained_models/",
# )
# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="nequip_organics_02.zip",
#     local_dir="pretrained_models/",
# )
# hf_hub_download(
#     repo_id="InstaDeepAI/mlip_models_organics_v2",
#     filename="esen_organics_02.zip",
#     local_dir="pretrained_models/",
# )

In [ ]:
from mlip.models import Visnet  # (1) we use ViSNet for this example.
from mlip.models.model_io import load_model_from_zip

organics_model_path = "pretrained_models/visnet_organics_02.zip"

# from mlip.models import Mace
# organics_model_path = "pretrained_models/mace_organics_02.zip"
# from mlip.models import Nequip
# organics_model_path = "pretrained_models/nequip_organics_02.zip"
# from mlip.models import Esen
# organics_model_path = "pretrained_models/esen_organics_02.zip"

force_field = load_model_from_zip(Visnet, organics_model_path)

# Derive a short label from the zip filename for use in plots
model_label = organics_model_path.rsplit("/", maxsplit=1)[-1].replace(".zip", "")

## 2. Inspect the initial structure

In [ ]:
import numpy as np
from ase.io import read as ase_read

atoms = ase_read("advanced_simulation/nacl_equil.xyz")

atomic_numbers = atoms.get_atomic_numbers()
na_idx = int(np.where(atomic_numbers == 11)[0][0])  # Na
cl_idx = int(np.where(atomic_numbers == 17)[0][0])  # Cl

initial_distance = np.linalg.norm(
    atoms.get_positions()[na_idx] - atoms.get_positions()[cl_idx]
)

print(f"Na index: {na_idx},  Cl index: {cl_idx}")
print(f"Initial Na–Cl distance: {initial_distance:.2f} Å")
print(f"Number of atoms: {len(atoms)}")
print(f"Cell: {atoms.get_cell().diagonal()} Å")

## 3. Define the collective variables (CVs)

Following Laio & Parrinello [1] and many others, we use two collective variables (CVs):

**CV 1: Na–Cl distance**: the primary reaction coordinate. It is ~2.7 Å in the contact
ion pair and ~5.0 Å in the solvent-separated state.

**CV 2: Coordination number of Na with water oxygens**: a differentiable approximation
to the integer count of O atoms within the first solvation shell of Na⁺, computed via
a rational switching function [1]:

$$\text{CN}(r) = \sum_{j \in \text{O}} \frac{1 - (r_{\text{Na},j}/r_0)^{n}}{1 - (r_{\text{Na},j}/r_0)^{m}}$$

As the ion pair dissociates, water molecules fill the vacancy left by Cl⁻, so
the Na–O coordination number increases from ~5 (CIP) to ~6 (SSIP). Without this
second CV, the free-energy surface would conflate configurations that are kinetically
distinct.

In [ ]:
from mlip.simulation.metadynamics.potential_terms import (
    CoordinationNumberCVConfig,
    DistanceCVConfig,
    DistanceWallConfig,
)

# CV 1: Na–Cl distance
cv_distance = DistanceCVConfig(atom_indices_1=[na_idx], atom_indices_2=[cl_idx])

# CV 2: coordination number of Na with water oxygens
# r0 = 3.15 Å is the reference distance (switching function value = 0.5 at r0)
# n=12, m=24 give a smooth step from 1 (r << r0) to 0 (r >> r0)
cv_coordnum = CoordinationNumberCVConfig(
    central_idx=na_idx,
    element="O",  # water oxygens
    r0=3.15,  # Å — first solvation shell of Na+ extends to ~3.2 Å
    nn=12,
    mm=24,
    d_max=5.0,  # Å — hard cutoff beyond the first shell
)

# Upper wall on distance: keep ions within one solvation-shell separation
upper_wall = DistanceWallConfig(
    atom_indices_1=[na_idx],
    atom_indices_2=[cl_idx],
    upper=7.0,  # Å
    kappa=150.0,  # eV/Å²
)

## 4. Configure the metadynamics simulation

Key parameters:

- **`bias_sigmas`**: Gaussian hill widths in CV units. 0.15 Å for the distance and
  0.2 (dimensionless) for the coordination number give good resolution of both basins.
- **`bias_factor`** (γ): at 300 K with γ = 15, the effective bias temperature is
  (γ−1)×T = 4200 K, sufficient to cross barriers of several kcal/mol.
- **`initial_height`**: 0.02 eV (~0.5 k_BT at 300 K).
- **`deposition_interval`**: a hill every 100 steps = 0.1 ps.

In [ ]:
from mlip.simulation.enums import MDIntegrator, SimulationType
from mlip.simulation.metadynamics.config import MetadynamicsConfig
from mlip.simulation.metadynamics.jax_md_metad_engine import (
    JaxMDMetadynamicsSimulationEngine,
)

metad_config = MetadynamicsConfig(
    bias_cvs=[cv_distance, cv_coordnum],
    bias_sigmas=[0.15, 0.2],  # Å, dimensionless
    bias_factor=15.0,
    initial_height=0.02,  # eV
    deposition_interval=100,  # (GPU run)
    # deposition_interval=50,  # (CPU run)
    max_gaussians=10000,
    walls=[upper_wall],
)

engine_config = JaxMDMetadynamicsSimulationEngine.Config(
    metadynamics_config=metad_config,
    simulation_type=SimulationType.MD,
    md_integrator=MDIntegrator.NVT_LANGEVIN,
    num_steps=250_000,  # 250 ps (GPU run)
    # num_steps=5000,  # 5 ps (CPU run)
    snapshot_interval=10,
    num_episodes=50,  # 5000 steps/episode (GPU run) or 100 (CPU run)
    timestep_fs=1.0,
    temperature_kelvin=300.0,
)

## 5. Run the simulation

**NOTE:** The GPU and CPU configs are each expected to take around **20–30 minutes** on respective hardware. GPU runtime tested on a 40GB NVIDIA H100.

In [ ]:
engine = JaxMDMetadynamicsSimulationEngine(atoms, force_field, engine_config)
engine.run()

## 6. Inspect the simulation state

The [`MetadynamicsSimulationState`](https://instadeepai.github.io/mlip/api_reference/simulation/simulation_state.html#mlip.simulation.metadynamics.states.MetadynamicsSimulationState) extends [`SimulationState`](https://instadeepai.github.io/mlip/api_reference/simulation/simulation_state.html#mlip.simulation.state.SimulationState) with:

| Field | Shape | Description |
|---|---|---|
| `bias_cv_values` | `(num_snapshots, num_cvs)` | Values of the CVs used by the bias potential at each snapshot |
| `bias_potential` | `(num_snapshots,)` | Total bias energy (eV) at each snapshot |
| `gaussian_centers` | `(num_hills, num_cvs)` | Hill positions along each CV |
| `gaussian_heights` | `(n_hills,)` | Hill heights (eV), rescaled by well-tempering |

In [ ]:
state = engine.state

cv_values_1 = state.bias_cv_values[:, 0]
cv_values_2 = state.bias_cv_values[:, 1]

print("Steps completed:", state.step)
print("Hills deposited:", (state.gaussian_heights != 0).sum())
print(
    f"Na–Cl distance range sampled: {cv_values_1.min():.2f} – {cv_values_1.max():.2f} Å"
)
print(
    "Na–O coordination number range sampled: "
    f"{cv_values_2.min():.2f} – {cv_values_2.max():.2f}"
)

## 7. Visualise the CV trajectories

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

dt = engine_config.snapshot_interval * engine_config.timestep_fs  # fs per snapshot
time_ps = np.arange(len(state.bias_cv_values)) * dt / 1000

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(time_ps, cv_values_1, lw=0.5, color="steelblue")
axes[0].axhline(2.7, ls="--", color="gray", lw=0.8, label="CIP (~2.7 Å)")
axes[0].axhline(5.0, ls="--", color="salmon", lw=0.8, label="SSIP (~5.0 Å)")
axes[0].set_ylabel("Na–Cl distance (Å)")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].plot(time_ps, cv_values_2, lw=0.5, color="seagreen")
axes[1].set_ylabel("Na–O coordination number")

axes[2].plot(time_ps, state.bias_potential, lw=0.5, color="coral")
axes[2].set_ylabel("Bias (eV)")
axes[2].set_xlabel("Time (ps)")

plt.tight_layout()
plt.show()

## 8. 2D free-energy surface

We reconstruct the 2D free-energy surface $F(d_{\text{NaCl}}, \text{CN}_{\text{Na–O}})$
from the accumulated Gaussian bias using the well-tempered relation:

$$F(\mathbf{s}) \approx -\frac{\gamma}{\gamma - 1} \, V_\text{bias}(\mathbf{s},\, t \to \infty)$$

In [ ]:
gamma = metad_config.bias_factor
sigma_1, sigma_2 = metad_config.bias_sigmas

mask = state.gaussian_heights != 0
centers_1 = state.gaussian_centers[mask, 0]
centers_2 = state.gaussian_centers[mask, 1]
heights = state.gaussian_heights[mask]

# 2D grid
cv_1_linspace = np.linspace(cv_values_1.min() - 0.1, cv_values_1.max() + 0.1, 150)
cv_2_linspace = np.linspace(cv_values_2.min() - 0.1, cv_values_2.max() + 0.1, 150)
grid_1, grid_2 = np.meshgrid(cv_1_linspace, cv_2_linspace)

bias_2d = np.sum(
    heights[:, None, None]
    * np.exp(
        -0.5 * ((grid_1[None] - centers_1[:, None, None]) / sigma_1) ** 2
        - 0.5 * ((grid_2[None] - centers_2[:, None, None]) / sigma_2) ** 2
    ),
    axis=0,
)

fes_2d = -(gamma / (gamma - 1)) * bias_2d
fes_2d -= fes_2d.min()
eV_to_kcalmol = 23.0609
fes_2d_kcal = fes_2d * eV_to_kcalmol

plt.figure(figsize=(8, 6))
levels = np.linspace(0, 10, 21)
cf = plt.contourf(
    cv_1_linspace, cv_2_linspace, fes_2d_kcal, levels=levels, cmap="RdYlBu_r"
)
plt.colorbar(cf, label="Free energy (kcal/mol)")
plt.contour(
    cv_1_linspace,
    cv_2_linspace,
    fes_2d_kcal,
    levels=levels,
    colors="k",
    linewidths=0.3,
    alpha=0.5,
)
plt.xlabel("Na–Cl distance (Å)")
plt.ylabel("Na–O coordination number")
plt.title("2D free-energy surface: NaCl dissociation in water")
plt.tight_layout()
plt.show()

## 9. 1D potential of mean force

Projecting onto the Na–Cl distance gives the 1D PMF, which should show the
CIP and SSIP basins with a desolvation barrier of ~2–3 kcal/mol, consistent
with classical force-field simulations and experiment.

In [ ]:
# Marginalise over CN: F(d) = -kT ln ∫ exp(-F(d,CN)/kT) dCN
from ase.units import kB

kT_kcal = kB * engine_config.temperature_kelvin * eV_to_kcalmol
pmf_1d = -kT_kcal * np.log(np.sum(np.exp(-fes_2d_kcal / kT_kcal), axis=0))
pmf_1d -= pmf_1d.min()

plt.figure(figsize=(8, 4))
plt.plot(cv_1_linspace, pmf_1d, color="steelblue", lw=2)
plt.axvline(2.7, ls="--", color="gray", lw=0.8, label="CIP")
plt.axvline(5.0, ls="--", color="salmon", lw=0.8, label="SSIP")
plt.xlabel("Na–Cl distance (Å)")
plt.ylabel("PMF (kcal/mol)")
plt.title("1D PMF: NaCl dissociation in water")
plt.legend()
plt.tight_layout()
plt.show()